# Experiment 01 on Colab: sentence-level baseline

Runs `experiments/01_sentence_baseline.py` on a Colab GPU. This notebook only sets up and calls
the script; all logic lives in the repo so local and Colab runs are identical.

1. **Runtime → Change runtime type → GPU** (T4 is fine).
2. Run cells top to bottom. Scores are written to Google Drive after every response, so if the
   session disconnects, re-run all cells and it resumes where it stopped.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Settings

In [ ]:
REPO = "https://github.com/vedantrajeshirke/agentic-hallucination-detection.git"
REF = "master"          # pin to a commit hash for a reported run
SPLIT = "dev"           # "dev" first (to tune the threshold), then "test"
PER_TASK = "50"         # responses per task; "" = whole split
MODEL = "large"
DRIVE_OUT = "/content/drive/MyDrive/agentic-hallucination/results/01_sentence_baseline"

## Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(DRIVE_OUT, exist_ok=True)

In [ ]:
%cd /content
!test -d repo || git clone -q $REPO repo
%cd /content/repo
!git fetch -q origin && git checkout -q $REF && git log -1 --oneline

In [ ]:
# Exact versions from uv.lock (first run downloads torch + CUDA libs, a few minutes)
!pip -q install uv
!uv sync -q --python 3.12
!uv run python -c "import torch; print(torch.__version__, 'CUDA:', torch.cuda.is_available())"

**If the cell above printed `CUDA: False`:** the locked torch build (CUDA 13) needs a newer GPU driver than this
Colab runtime has. Run the next cell instead: it keeps every locked package except torch, which comes from
Colab's preinstalled CUDA build. The run config records the torch version actually used. Otherwise skip it.


In [ ]:
# FALLBACK ONLY: use Colab's own torch with the rest of the locked versions
# !uv venv -q --python 3.12 --system-site-packages --allow-existing
# !uv export -q --no-hashes --no-emit-project | grep -v -E '^(torch|nvidia-|triton)' > /tmp/req.txt
# !uv pip install -q -r /tmp/req.txt
# !uv run --no-sync python -c "import torch; print(torch.__version__, 'CUDA:', torch.cuda.is_available())"
# then add --no-sync after `uv run` in the cells below


## Run (resumable)

In [ ]:
per_task = f"--per-task {PER_TASK}" if PER_TASK else ""
!uv run python experiments/01_sentence_baseline.py run --split $SPLIT $per_task --model $MODEL --fp16 --batch-size 32 --out-dir "$DRIVE_OUT"

## Evaluate

Tune the threshold on the **dev** scores file, report on the **test** file. Tuning and reporting
on the same file is only a smoke check, and the summary says so.

In [ ]:
!ls -1 "$DRIVE_OUT"

In [ ]:
TUNE = f"{DRIVE_OUT}/scores_dev_n50_large_per_passage.jsonl"
REPORT = TUNE  # replace with the test scores file once it exists
!uv run python experiments/01_sentence_baseline.py evaluate --tune "$TUNE" --report "$REPORT"